# Phase 5 - Retrieval profiles, BM25, reranking và bounded context

Notebook này giới thiệu retrieval pipeline local của Hue Foods RAG MVP theo
`guides/phase_5_retrieval_profiles_reranking.md`. Notebook import backend runtime
(`RetrievalService`, `ContextBuilder`, startup) và không duplicate runtime logic.

Ba profile chính thức:

| Profile | Dense Qdrant | BM25 fusion | Reranker | Output |
|---|---|---|---|---|
| `dense_only` | top 10 | không | không | 10 documents |
| `hybrid_no_rerank` | top 30 | 0.6/0.4 | không | 10 documents |
| `hybrid_rerank` | top 30 | 0.6/0.4 | local MiniLM | 5 documents |

Default mode của notebook dùng fake dependencies: không mở Qdrant, không tải
model và không gọi bất kỳ API nào. Real local mode nằm trong cell cuối và phải
bật explicit environment guard.

In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
print(f"backend on path: {sys.path[0]}")

## 1. Cấu hình

Config canonical nằm trong `backend/config/settings.yaml`. Các giá trị quan
trọng cho Phase 5: retrieval depth, candidate depth, fusion weights và context
budget; reranking model/device/top_k.

In [ ]:
from core.settings_loader import load_settings

settings = load_settings()
retrieval = settings["retrieval"]
reranking = settings["reranking"]
print("active_profile:", settings["active_profile"])
print("retrieval.top_k:", retrieval["top_k"])
print("retrieval.candidate_multiplier:", retrieval["candidate_multiplier"])
print("dense/bm25 weights:", retrieval["dense_weight"], retrieval["bm25_weight"])
print("context limits:", retrieval["max_context_documents"], "docs /", retrieval["max_context_characters"], "chars")
print("reranking.model:", reranking["model"])
print("reranking.device:", reranking["device"])
print("reranking.top_k:", reranking["top_k"])

## 2. Fake dependencies (default mode)

Default mode không được mở Qdrant, tải model hoặc gọi external API. Vì vậy
notebook xây fake embedder và fake client theo đúng contract backend:

- Fake embedder: `model_id`, `dimension`, `embed_query` deterministic.
- Fake client: `collection_exists`, `get_collection`, `count`, `scroll`
  (bounded batches, `with_vectors=False`) và `query_points` cho named vector
  `dense`, với 572 payloads hợp lệ - đúng số lượng canonical của Phase 2.

Startup `build_retrieval_stack` vẫn chạy đầy đủ verification (schema, count,
unique `chunk_id`, non-empty text, embedding model, corpus fingerprint) trên
fake này, nên đường đi chính thức được demo mà không cần infrastructure.

In [ ]:
from types import SimpleNamespace

from qdrant_client import models


class FakeEmbedder:
    """Deterministic fake embedder matching the BaseEmbedder contract."""

    def __init__(self, model_id, dimension):
        self._model_id = model_id
        self._dimension = dimension

    @property
    def model_id(self):
        return self._model_id

    @property
    def dimension(self):
        return self._dimension

    def embed_query(self, query):
        return [0.1 + (len(query) % 7) * 0.01] * self._dimension

    def embed_documents(self, texts):
        return [self.embed_query(text) for text in texts]


def make_payloads(count, model, dimension):
    """Deterministic 572 fake payloads matching the Phase 4 payload contract."""
    payloads = []
    for i in range(count):
        payloads.append(
            {
                "chunk_id": f"foods/restaurants/quan{i:03d}.md|Tóm tắt|0",
                "text": f"Món ăn ở quán số {i}: bún bò Huế, cơm hến, chè.",
                "source": f"foods/restaurants/quan{i:03d}.md",
                "title": "Quán",
                "section": "Tóm tắt",
                "category": "foods",
                "subcategory": "restaurants",
                "chunk_type": "section",
                "embedding_model": model,
                "embedding_dimension": dimension,
            }
        )
    return payloads


class FakeClient:
    """In-memory Qdrant fake: schema info, count, bounded scroll, dense query."""

    def __init__(self, payloads, dimension):
        self._payloads = payloads
        vectors = {"dense": SimpleNamespace(size=dimension, distance=models.Distance.COSINE)}
        sparse = {"sparse": SimpleNamespace(index=object())}
        self._info = SimpleNamespace(
            config=SimpleNamespace(
                params=SimpleNamespace(vectors=vectors, sparse_vectors=sparse)
            )
        )

    def collection_exists(self, name):
        return True

    def get_collection(self, name):
        return self._info

    def count(self, name, exact=True):
        return SimpleNamespace(count=len(self._payloads))

    def scroll(self, name, limit, offset=None, with_payload=True, with_vectors=False, timeout=None):
        start = 0 if offset is None else offset
        batch = self._payloads[start : start + limit]
        next_offset = (
            start + len(batch) if start + len(batch) < len(self._payloads) else None
        )
        return [SimpleNamespace(payload=p) for p in batch], next_offset

    def query_points(self, collection_name, query, using=None, limit=None, **kwargs):
        points = [
            SimpleNamespace(id=p["chunk_id"], score=1.0 - i * 0.001, payload=p)
            for i, p in enumerate(self._payloads)
        ]
        return SimpleNamespace(points=points[:limit])


embedding = settings["embedding"]
fake_embedder = FakeEmbedder(
    model_id=embedding["model"], dimension=embedding["vector_size"]
)
fake_payloads = make_payloads(
    count=572, model=embedding["model"], dimension=embedding["vector_size"]
)
fake_client = FakeClient(fake_payloads, dimension=embedding["vector_size"])
print("fake embedder:", fake_embedder.model_id, fake_embedder.dimension)
print("fake payloads:", len(fake_payloads))

## 3. Profile `dense_only`

Flow: query -> validate -> E5 query embedding (prefix `query: `) -> Qdrant named
vector `dense` top 10 -> deterministic ordering.

Profile này không scroll corpus, không fit BM25 và không load reranker. Score
metadata chỉ chứa các field của stage đã chạy: `dense_score`, `embedding_model`,
`retrieval_profile`, `retrieval_rank`.

In [ ]:
from core.startup import build_retrieval_stack
from retrieval.service import RetrievalService

settings_dense = load_settings()
settings_dense["active_profile"] = "dense_only"
stack_dense = build_retrieval_stack(
    settings_dense, client=fake_client, embedder=fake_embedder
)
print("snapshot active_profile:", stack_dense.snapshot.active_profile)
print("bm25_ready:", stack_dense.snapshot.bm25_ready)
print("reranker_ready:", stack_dense.snapshot.reranker_ready)
print("corpus_fingerprint:", stack_dense.snapshot.corpus_fingerprint)

service_dense = RetrievalService(
    stack_dense, rerank_top_k=settings["reranking"]["top_k"]
)
docs_dense = service_dense.search("cơm hến ở Huế")
print("documents:", len(docs_dense))
print("score fields of top-1:", sorted(docs_dense[0].metadata))

## 4. Profile `hybrid_no_rerank`

Flow: query -> E5 embedding -> Qdrant dense top 30 -> BM25 score trên 30 dense
candidates -> min-max normalize dense và BM25 độc lập -> fusion
`0.6 * normalized_dense + 0.4 * normalized_bm25` -> deterministic top 10.

BM25 được fit một lần trên toàn bộ 572 chunk texts lúc startup, không fit mỗi
request và không query named sparse vector của Qdrant. Metadata có thêm
`bm25_score`, `normalized_dense_score`, `normalized_bm25_score`, `hybrid_score`;
`RetrievedDocument.score` là hybrid score.

In [ ]:
settings_hybrid = load_settings()
settings_hybrid["active_profile"] = "hybrid_no_rerank"
stack_hybrid = build_retrieval_stack(
    settings_hybrid, client=fake_client, embedder=fake_embedder
)
print("bm25_ready:", stack_hybrid.snapshot.bm25_ready)
print("reranker_ready:", stack_hybrid.snapshot.reranker_ready)
print("corpus_fingerprint:", stack_hybrid.snapshot.corpus_fingerprint)

service_hybrid = RetrievalService(
    stack_hybrid, rerank_top_k=settings["reranking"]["top_k"]
)
docs_hybrid = service_hybrid.search("cơm hến ở Huế")
print("documents:", len(docs_hybrid))
for doc in docs_hybrid[:3]:
    print(doc.metadata["chunk_id"], "| hybrid=", round(doc.score, 4),
          "| dense=", round(doc.metadata["dense_score"], 4),
          "| norm_dense=", round(doc.metadata["normalized_dense_score"], 4),
          "| bm25=", round(doc.metadata["bm25_score"], 4),
          "| norm_bm25=", round(doc.metadata["normalized_bm25_score"], 4))

## 5. Profile `hybrid_rerank`

Flow: cùng hybrid pipeline (cùng candidate depth và fusion) -> hybrid top 10 ->
local CrossEncoder `cross-encoder/ms-marco-MiniLM-L-6-v2` score đúng 10 pairs
-> deterministic top 5.

Default mode dùng fake scorer vì không được tải model trong notebook; real mode
(dòng cuối) mới dùng MiniLM và cần local cache. Metadata có thêm
`rerank_score` và `reranker_model`; `RetrievedDocument.score` là rerank score.

In [ ]:
from reranking.reranker import ScorerReranker


def fake_scorer(query, documents):
    """Deterministic fake rerank scores; never loads a model."""
    return [round(0.9 - i * 0.04, 2) for i in range(len(documents))]


fake_reranker = ScorerReranker(scorer=fake_scorer, model_id="fake/demo-reranker")

settings_rerank = load_settings()
settings_rerank["active_profile"] = "hybrid_rerank"
stack_rerank = build_retrieval_stack(
    settings_rerank, client=fake_client, embedder=fake_embedder, reranker=fake_reranker
)
service_rerank = RetrievalService(
    stack_rerank, rerank_top_k=settings["reranking"]["top_k"]
)
docs_rerank = service_rerank.search("cơm hến ở Huế")
print("pre-rerank documents:", len(docs_hybrid), "| reranked documents:", len(docs_rerank))
for doc in docs_rerank:
    print(doc.metadata["chunk_id"], "| rerank=", doc.score,
          "| model=", doc.metadata["reranker_model"])

## 6. Whole-chunk bounded context

`ContextBuilder` ghép whole chunks thành context có giới hạn (tối đa 5 documents
và 3.000 characters, tính cả source label và separator). Chunk tiếp theo không
vừa thì dừng; không truncate chunk, không cắt bảng Markdown; giữ rank order và
trả source mapping kèm context.

In [ ]:
from retrieval.context_builder import ContextBuilder

builder = ContextBuilder(
    max_documents=settings["retrieval"]["max_context_documents"],
    max_characters=settings["retrieval"]["max_context_characters"],
)
result = builder.build(docs_dense)
print("context length:", len(result.context), "characters")
print("sources:", len(result.sources))
print("---")
print(result.context[:240])
print("---")
print("source mapping[0]:", result.sources[0])

## 7. Typed errors

Retrieval không che dấu lỗi rồi trả danh sách rỗng. Lỗi được phân loại:
`InvalidQueryError` (query rỗng/whitespace), `RetrievalConfigurationError`
(profile/config sai), `ComponentNotReadyError` (component thiếu hoặc snapshot
stale), `RetrievalDependencyError` (embedder/Qdrant/model lỗi). Chỉ khi
retrieval chạy thành công nhưng không có candidate mới trả `[]`.

In [ ]:
from core.schema import ComponentNotReadyError, InvalidQueryError
from core.startup import RetrievalStack

try:
    service_dense.search("   ")
except InvalidQueryError as exc:
    print("InvalidQueryError:", exc)

try:
    RetrievalService(
        RetrievalStack(snapshot=stack_dense.snapshot),
        rerank_top_k=settings["reranking"]["top_k"],
    )
except ComponentNotReadyError as exc:
    print("ComponentNotReadyError:", exc)

## 8. Real local mode (opt-in)

Real mode chỉ chạy khi user set `HUE_RAG_QDRANT_REAL=1` và đã có:

- Qdrant local đang chạy với collection `hue_foods_e5_small_384` (572 points).
- Local model cache cho E5 và MiniLM (không download tự động).

Real mode chỉ đọc collection, không reset/reindex, và không gọi OpenRouter hoặc
paid API. Latency gate p95 (warm-up 1 lượt, 20 lượt rerank 10 pairs, không quá
3 giây) là real validation cần approval riêng - không chạy trong notebook này.

In [ ]:
import os

if os.environ.get("HUE_RAG_QDRANT_REAL") == "1":
    from retrieval.service import build_service

    service_real = build_service()
    print("profile:", service_real.active_profile)
    print("snapshot:", service_real.snapshot)
    # Không chạy query ở đây: cần Qdrant local + model cache + approval riêng.
else:
    print("Skipped: real mode is off. Set HUE_RAG_QDRANT_REAL=1 only when")
    print("Qdrant local is up, E5/MiniLM are cached and the user approved.")

## Tóm tắt

- Ba profile chạy đúng stage cần thiết; `dense_only` độc lập khi BM25/reranker
  chưa được khởi tạo.
- Score metadata stage-conditional: không tạo field giả cho stage không chạy.
- Context whole-chunk, bounded và giữ source mapping theo rank.
- Lỗi typed, không silent fallback, không tự chuyển profile.
- Để kiểm tra lại: chạy lại toàn bộ notebook (default mode an toàn), hoặc chạy
  tests trong `backend/`:
  `uv run python -m pytest tests/test_bm25.py tests/test_retrieval_service.py tests/test_reranker.py tests/test_context_builder.py tests/test_startup.py -q --tb=short`